# Package

In [1]:
# ============================================================
# 1) Core Python & Paths
# ============================================================
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from dateutil.relativedelta import relativedelta

# Project root (notebook dans /notebooks)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# 2) Data & Utils (Feast + custom utils)
# ============================================================
import pandas as pd
import numpy as np

from utils import load_wide_from_feast, build_unrate_exog_dataset
from experiment_utils import to_wide_from_oos, make_mae_dm_pivot

# ============================================================
# 3) Forecasting (MLForecast + Models)
# ============================================================
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals
from sklearn.linear_model import Ridge
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV

# ============================================================
# 4) Evaluation & Statistical Tests
# ============================================================
from scipy.stats import ttest_rel
from statsmodels.stats.contingency_tables import mcnemar

# ============================================================
# 5) Visualization
# ============================================================
from utilsforecast.plotting import plot_series
from IPython.display import IFrame, display

# ============================================================
# 6) Experiment Tracking (MLflow)
# ============================================================
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.linear_model import LinearRegression

# Importation des données

In [2]:
series_ids = [
    "BUSLOANS","CPIAUCSL","DPCERA3M086SBEA","INDPRO",
    "M2SL","OILPRICEX","RPI","SP500","TB3MS","UNRATE","USREC",
]

START = "1960-01-01"
END   = "2025-08-01"

In [3]:
# ============================================================
# 1) WIDE dataset (df) depuis Feast
# ============================================================
import pandas as pd

LAG = 6  # ✅ tout lag = 12 (aucune variable contemporaine)

df = load_wide_from_feast(
    "stationary_value:value",
    series_ids,
    start=START,
    end=END
)

# ============================================================
# 2) Convert WIDE -> LONG (obligatoire pour build_unrate_exog_dataset)
# ============================================================
df_stationary = (
    df.reset_index()
      .melt(id_vars="date", var_name="series_id", value_name="value")
)

# ============================================================
# 3) Build target + exog + MLForecast format
# ============================================================
df_model, ts_lr, exog_cols = build_unrate_exog_dataset(df_stationary)

# ============================================================
# 4) ✅ TOUTES les exog laggées de 12 mois (aucune contemporaine)
#    - on crée BUSLOANS_lag12, CPIAUCSL_lag12, ...
#    - on supprime BUSLOANS, CPIAUCSL, ... (contemporaines)
# ============================================================
ts_lr = ts_lr.sort_values(["unique_id", "ds"]).copy()

exog_cols_lag12 = []
for c in exog_cols:
    new_c = f"{c}_lag{LAG}"
    ts_lr[new_c] = ts_lr.groupby("unique_id")[c].shift(LAG)
    exog_cols_lag12.append(new_c)

# supprimer les exog contemporaines
ts_lr = ts_lr.drop(columns=exog_cols)

# mettre à jour la liste exog utilisée par MLForecast
exog_cols = exog_cols_lag12

# drop lignes où les lag12 n'existent pas (12 premiers mois)
ts_lr = ts_lr.dropna(subset=["y"] + exog_cols).reset_index(drop=True)

# ============================================================
# 5) Prints / checks
# ============================================================
print("df (wide) shape:", df.shape)
print("df_stationary (long) shape:", df_stationary.shape)
print("df_model shape:", df_model.shape)
print("ts_lr shape (after lag12):", ts_lr.shape)
print("Exog cols (lag12):", exog_cols)

print("\nPreview:")
print(ts_lr[["unique_id", "ds", "y"] + exog_cols[:5]].head(15))

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df (wide) shape: (788, 11)
df_stationary (long) shape: (8668, 3)
df_model shape: (788, 12)
ts_lr shape (after lag12): (782, 13)
Exog cols (lag12): ['BUSLOANS_lag6', 'CPIAUCSL_lag6', 'DPCERA3M086SBEA_lag6', 'INDPRO_lag6', 'M2SL_lag6', 'OILPRICEX_lag6', 'RPI_lag6', 'SP500_lag6', 'TB3MS_lag6', 'USREC_lag6']

Preview:
   unique_id         ds    y  BUSLOANS_lag6  CPIAUCSL_lag6  \
0     UNRATE 1960-07-01  0.4       0.011578      -0.006156   
1     UNRATE 1960-08-01  0.4       0.011905      -0.003767   
2     UNRATE 1960-09-01  0.0      -0.008356      -0.005455   
3     UNRATE 1960-10-01  0.4      -0.009098       0.005090   
4     UNRATE 1960-11-01  0.3      -0.000359       0.003383   
5     UNRATE 1960-12-01  1.3       0.014620       0.006777   
6     UNRATE 1961-01-01  1.4      -0.000611      -0.005433   
7     UNRATE 1961-02-01  2.1      -0.016888      -0.004074   
8     UNRATE 1961-03-01  1.

d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\ML Experiment\utils.py:171: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


# Model settings

In [4]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Callable, Dict, Any, Optional, List, Iterable

from dateutil.relativedelta import relativedelta
from sklearn.model_selection import ParameterSampler, ParameterGrid
from sklearn.metrics import mean_absolute_error

from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals


# -----------------------------
# Helpers temps
# -----------------------------
def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

def _slice_cv_block(ts, cutoff_start, n_windows, h):
    cutoff_end = cutoff_start + relativedelta(months=n_windows - 1)
    ds_end = cutoff_end + relativedelta(months=h)
    return ts[ts["ds"] <= ds_end].copy(), cutoff_end, ds_end

In [5]:
# -----------------------------
# Spec modèle (tu ajoutes juste ici)
# -----------------------------
@dataclass
class ModelSpec:
    name: str                         # "LR", "RIDGE", "LGBM", etc.
    build_mlf: Callable[[str, Dict[str, Any]], MLForecast]  # (freq, params)->MLForecast
    pred_col: str                     # colonne de forecast dans cv (ex "LR")
    tunable: bool = False
    param_space: Optional[Dict[str, Iterable[Any]]] = None
    search: str = "random"            # "random"|"grid"
    n_iter: int = 50                  # si random
    tune_cv_windows: int = 6
    tune_every_months: int = 36
    use_conformal_in_tune: bool = False
    fixed_params: Optional[Dict[str, Any]] = None


In [6]:
# -----------------------------
# Tuner générique (pour tous les modèles)
# -----------------------------
def _tune_on_train(
    ts_train: pd.DataFrame,
    *,
    spec: ModelSpec,
    freq: str,
    h: int,
    levels: List[int],
    seed: int,
    pi_windows_cap: int,
    min_train_n: Optional[int] = None,
) -> tuple[Optional[Dict[str, Any]], float]:

    if min_train_n is not None and len(ts_train) < int(min_train_n):
        return None, float("nan")

    if not spec.tunable:
        return (spec.fixed_params or {}), float("nan")

    if not spec.param_space:
        raise ValueError(f"{spec.name}: tunable=True mais param_space=None")

    # conformal au tuning (souvent OFF)
    pi_tune = (
        PredictionIntervals(h=h, n_windows=min(int(spec.tune_cv_windows), int(pi_windows_cap)),
                            method="conformal_distribution")
        if spec.use_conformal_in_tune else None
    )

    # générateur d'essais
    if spec.search == "grid":
        sampler = ParameterGrid(spec.param_space)
    else:
        sampler = ParameterSampler(spec.param_space, n_iter=int(spec.n_iter), random_state=int(seed))

    best_params = None
    best_score = np.inf

    for params in sampler:
        params = dict(params)

        mlf = spec.build_mlf(freq, params)
        cv = mlf.cross_validation(
            df=ts_train,
            h=h,
            step_size=1,
            n_windows=int(spec.tune_cv_windows),
            prediction_intervals=pi_tune,
            level=list(levels) if pi_tune is not None else None,
            fitted=False,
            static_features=[],
            dropna=True,
        )
        score = mean_absolute_error(cv["y"], cv[spec.pred_col])

        if score < best_score:
            best_score = float(score)
            best_params = params

    return best_params, float(best_score)


In [7]:
# -----------------------------
# Runner multi-modèles : tu ajoutes juste un spec dans la liste
# -----------------------------
def run_backtesting_generic(
    ts: pd.DataFrame,
    *,
    model_specs: List[ModelSpec],
    freq: str,
    h: int,
    exp_start,
    exp_end,
    step_size: int,
    pi_windows: int,
    levels: List[int],
    seed: int = 0,
    min_train_n: Optional[int] = None,
) -> tuple[pd.DataFrame, Dict[str, Any]]:

    # run modèle par modèle puis merge
    bkts = []
    metas = {}

    for spec in model_specs:
        bkt_m, meta_m = backtest_one_model_tune_blocks(
            ts,
            spec=spec,
            freq=freq,
            h=h,
            exp_start=exp_start,
            exp_end=exp_end,
            step_size=step_size,
            pi_windows=pi_windows,
            levels=levels,
            seed=seed,
            min_train_n=min_train_n,
        )
        metas[spec.name] = meta_m
        if len(bkt_m):
            bkts.append(bkt_m)

    if not bkts:
        return pd.DataFrame(), {"error": "aucun modèle n’a produit de backtest", "metas": metas}

    # merge wide sur clés
    keys = ["unique_id", "ds", "cutoff", "y"]
    bkt_all = bkts[0].copy()

    for b in bkts[1:]:
        # éviter collisions si certains champs internes existent
        keep_cols = [c for c in b.columns if c not in bkt_all.columns or c in keys]
        bkt_all = bkt_all.merge(b[keep_cols], on=keys, how="outer")

    bkt_all = bkt_all.sort_values(["unique_id", "ds", "cutoff"]).reset_index(drop=True)

    # ✅ 1 ligne par ds : dernier cutoff
    bkt_final = (
        bkt_all.sort_values(["unique_id", "ds", "cutoff"])
               .groupby(["unique_id", "ds"], as_index=False)
               .tail(1)
               .reset_index(drop=True)
    )

    meta = {"metas": metas}
    return bkt_final, meta

In [8]:
# -----------------------------
# Backtesting générique avec retrain+tuning par blocs
# -> renvoie bkt wide pour 1 modèle (colonne spec.pred_col + PI)
# -----------------------------
def backtest_one_model_tune_blocks(
    ts: pd.DataFrame,
    *,
    spec: ModelSpec,
    freq: str,
    h: int,
    exp_start,
    exp_end,
    step_size: int,
    pi_windows: int,
    levels: List[int],
    seed: int = 0,
    min_train_n: Optional[int] = None,
) -> tuple[pd.DataFrame, Dict[str, Any]]:

    ts = ts.copy()
    ts["ds"] = (
        pd.to_datetime(ts["ds"], errors="coerce")
          .dt.to_period("M")
          .dt.to_timestamp(how="start")
          .dt.normalize()
    )
    if ts["ds"].isna().any():
        bad = ts[ts["ds"].isna()].head()
        raise ValueError(f"Dates 'ds' invalides après parsing. Exemples:\n{bad}")

    exp_start = _ensure_ms(exp_start)
    exp_end   = _ensure_ms(exp_end)

    cutoff_start_all = exp_start - relativedelta(months=h)
    cutoff_end_all   = exp_end   - relativedelta(months=h)
    total_partitions = _n_windows_monthly(cutoff_start_all, cutoff_end_all)

    # anti-fuite
    ts = ts[ts["ds"] <= exp_end].copy()

    # conformal au backtest (ici ON)
    pi = PredictionIntervals(h=h, n_windows=int(pi_windows), method="conformal_distribution")

    # blocs de retrain/tuning
    blocks = []
    remaining = int(total_partitions)
    cur = cutoff_start_all
    while remaining > 0:
        n_win = min(int(spec.tune_every_months), remaining)
        blocks.append((cur, n_win))
        cur = cur + relativedelta(months=n_win)
        remaining -= n_win

    all_bkts = []
    params_history = []
    tune_history = []

    for block_idx, (cutoff_start_blk, n_windows_blk) in enumerate(blocks, start=1):
        ts_blk, _, _ = _slice_cv_block(ts, cutoff_start_blk, int(n_windows_blk), h)

        ts_train_for_tune = ts[ts["ds"] <= cutoff_start_blk].copy()
        if min_train_n is not None and len(ts_train_for_tune) < int(min_train_n):
            continue

        best_params, tune_mae = _tune_on_train(
            ts_train_for_tune,
            spec=spec,
            freq=freq,
            h=h,
            levels=levels,
            seed=seed,
            pi_windows_cap=int(pi_windows),
            min_train_n=min_train_n,
        )
        if best_params is None:
            continue

        params_history.append({
            "model": spec.name,
            "block": block_idx,
            "cutoff_start": cutoff_start_blk,
            "n_windows": int(n_windows_blk),
            **best_params,
        })
        tune_history.append({
            "model": spec.name,
            "block": block_idx,
            "cutoff_start": cutoff_start_blk,
            "tune_mae": float(tune_mae),
        })

        mlf_blk = spec.build_mlf(freq, best_params)

        bkt_blk = mlf_blk.cross_validation(
            df=ts_blk,
            h=h,
            step_size=int(step_size),
            n_windows=int(n_windows_blk),
            prediction_intervals=pi,
            level=list(levels),
            fitted=True,
            static_features=[],
            dropna=True,
        )
        bkt_blk[f"{spec.name}_tune_block"] = block_idx
        bkt_blk[f"{spec.name}_tune_mae"] = float(tune_mae)

        all_bkts.append(bkt_blk)

    if not all_bkts:
        return pd.DataFrame(), {"error": f"{spec.name}: aucun bloc produit"}

    bkt = pd.concat(all_bkts, ignore_index=True)

    # filtre exp exact
    bkt = bkt[(bkt["ds"] >= exp_start) & (bkt["ds"] <= exp_end)].copy()
    bkt = bkt.sort_values(["unique_id", "ds", "cutoff"]).reset_index(drop=True)

    meta = dict(
        model=spec.name,
        h=int(h),
        step_size=int(step_size),
        exp_start=exp_start,
        exp_end=exp_end,
        cutoff_start=cutoff_start_all,
        cutoff_end=cutoff_end_all,
        partitions=int(total_partitions),
        pi_windows=int(pi_windows),
        tune_every_months=int(spec.tune_every_months),
        tune_cv_windows=int(spec.tune_cv_windows),
        tunable=bool(spec.tunable),
        search=spec.search,
        n_iter=int(spec.n_iter),
        use_conformal_in_tune=bool(spec.use_conformal_in_tune),
        param_space=spec.param_space,
        params_history=params_history,
        tune_history=tune_history,
    )
    return bkt, meta


In [9]:
# ============================================================
# Builders modèles
# ============================================================

def build_lr(freq, params):
    return MLForecast(models={"LR": LinearRegression()}, freq=freq, lags=[], date_features=[])

def build_ridge(freq, params):
    alpha = params.get("alpha", 1.0)
    return MLForecast(models={"RIDGE": Ridge(alpha=alpha)}, freq=freq, lags=[], date_features=[])

LGBM_BASE = dict(random_state=0, n_jobs=-1, verbosity=-1, objective="regression", metric="mae")

def build_lgbm(freq, params):
    p = dict(LGBM_BASE)
    p.update(params)
    return MLForecast(models={"LGBM": LGBMRegressor(**p)}, freq=freq, lags=[], date_features=[])

# ============================================================
# Model Specs (LR + RIDGE + LGBM)
# ============================================================

MODEL_SPECS = [
    ModelSpec(
        name="LR",
        build_mlf=build_lr,
        pred_col="LR",
        tunable=False,
        fixed_params={},
    ),
    ModelSpec(
        name="RIDGE",
        build_mlf=build_ridge,
        pred_col="RIDGE",
        tunable=True,
        param_space={"alpha": [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]},
        search="grid",
        tune_every_months=36,
    ),
    ModelSpec(
        name="LGBM",
        build_mlf=build_lgbm,
        pred_col="LGBM",
        tunable=True,
        param_space={
            "n_estimators": [20, 50, 100],
            "max_depth": [2, 3, 5],
            "num_leaves": [7, 15, 31],
        },
        search="random",
        n_iter=12,
        tune_every_months=36,
    ),
]

In [10]:
# 2) Run LR + RIDGE (avec retrain/tuning par blocs pour Ridge)
H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]
FREQ = "MS"
EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")
SEED = 42

bkt_models_final, meta_models = run_backtesting_generic(
    ts=ts_lr,
    model_specs=MODEL_SPECS,   # ← ta liste avec LR + RIDGE + LGBM
    freq=FREQ,
    h=H,
    exp_start=EXP_START,
    exp_end=EXP_END,
    step_size=STEP_SIZE,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
    seed=SEED,
    min_train_n=36,
)

print("✅ rows:", len(bkt_models_final))
print("✅ columns (head):", bkt_models_final.columns.tolist()[:25])

# vérifier blocs Ridge
print("✅ Ridge blocks:", len(meta_models["metas"]["RIDGE"].get("params_history", [])))

# vérifier blocs LGBM
print("✅ LGBM blocks:", len(meta_models["metas"]["LGBM"].get("params_history", [])))

bkt_models_final.head()

✅ rows: 428
✅ columns (head): ['unique_id', 'ds', 'cutoff', 'y', 'LR', 'LR-lo-95', 'LR-hi-95', 'LR_tune_block', 'LR_tune_mae', 'RIDGE', 'RIDGE-lo-95', 'RIDGE-hi-95', 'RIDGE_tune_block', 'RIDGE_tune_mae', 'LGBM', 'LGBM-lo-95', 'LGBM-hi-95', 'LGBM_tune_block', 'LGBM_tune_mae']
✅ Ridge blocks: 12
✅ LGBM blocks: 12


,unique_id,ds,cutoff,y,LR,LR-lo-95,LR-hi-95,LR_tune_block,LR_tune_mae,RIDGE,RIDGE-lo-95,RIDGE-hi-95,RIDGE_tune_block,RIDGE_tune_mae,LGBM,LGBM-lo-95,LGBM-hi-95,LGBM_tune_block,LGBM_tune_mae
0,UNRATE,1990-01-01,1989-12-01,0.0,0.293094,-0.075968,0.662156,1,NaN,-0.284023,-0.766392,0.198347,1,0.407871,-0.082123,-0.278170,0.113923,1,0.26636
1,UNRATE,1990-02-01,1990-01-01,0.1,0.022359,-0.432703,0.477422,1,NaN,-0.310885,-0.854670,0.232899,1,0.407871,0.080220,-0.468733,0.629172,1,0.26636
2,UNRATE,1990-03-01,1990-02-01,0.2,0.003990,-0.797507,0.805487,1,NaN,-0.348619,-0.959538,0.262301,1,0.407871,-0.213265,-1.421804,0.995274,1,0.26636
3,UNRATE,1990-04-01,1990-03-01,0.2,-0.188218,-0.849793,0.473358,1,NaN,-0.391875,-0.984998,0.201247,1,0.407871,0.098855,-0.543136,0.740846,1,0.26636
4,UNRATE,1990-05-01,1990-04-01,0.2,-0.144925,-0.715670,0.425821,1,NaN,-0.397146,-0.911522,0.117230,1,0.407871,0.079659,-0.397057,0.556376,1,0.26636


# Analyze

In [12]:
import pandas as pd
import numpy as np

df = bkt_models_final.copy()
df["ds"] = pd.to_datetime(df["ds"], errors="coerce")

# enlever timezone si besoin
if pd.api.types.is_datetime64tz_dtype(df["ds"]):
    df["ds"] = df["ds"].dt.tz_convert(None)

df = df.dropna(subset=["ds"]).reset_index(drop=True)

# bins / labels
bins = pd.to_datetime(["1990-01-01","2000-01-01","2009-01-01","2020-01-01","2025-09-01"])
labels = ["1990-1999", "2000-2008", "2009-2019", "2020-end"]

df["partition"] = pd.cut(df["ds"], bins=bins, labels=labels, right=False, include_lowest=True)
df = df.dropna(subset=["partition"]).reset_index(drop=True)

print(df["partition"].value_counts().sort_index())

partition
1990-1999    120
2000-2008    108
2009-2019    132
2020-end      68
Name: count, dtype: int64


C:\Users\Mita\AppData\Local\Temp\ipykernel_1948\2146780136.py:8: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df["ds"]):


In [14]:
import numpy as np
import pandas as pd

# -------------------------------------------------
# 1) Copier le backtest final
# -------------------------------------------------
df = bkt_models_final.copy()   # ← ton backtest multi-modèles

# -------------------------------------------------
# 2) Ajouter les partitions si pas encore faites
# -------------------------------------------------
df["ds"] = pd.to_datetime(df["ds"], errors="coerce")

bins = pd.to_datetime([
    "1990-01-01",
    "2000-01-01",
    "2009-01-01",
    "2020-01-01",
    "2025-09-01"
])

labels = ["1990-1999", "2000-2008", "2009-2019", "2020-end"]

df["partition"] = pd.cut(
    df["ds"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

df = df.dropna(subset=["partition"]).reset_index(drop=True)

# -------------------------------------------------
# 3) Erreurs absolues
# -------------------------------------------------
df["AE_LR"]    = (df["y"] - df["LR"]).abs()
df["AE_RIDGE"] = (df["y"] - df["RIDGE"]).abs()
df["AE_LGBM"]  = (df["y"] - df["LGBM"]).abs()

# -------------------------------------------------
# 4) MAE global (Ensemble)
# -------------------------------------------------
mae_global = {
    "LR": df["AE_LR"].mean(),
    "RIDGE": df["AE_RIDGE"].mean(),
    "LGBM": df["AE_LGBM"].mean(),
}

# -------------------------------------------------
# 5) MAE par partition
# -------------------------------------------------
mae_part = (
    df.groupby("partition")[["AE_LR","AE_RIDGE","AE_LGBM"]]
      .mean()
)

mae_part.columns = ["LR","RIDGE","LGBM"]

# -------------------------------------------------
# 6) Format final (models en lignes)
# -------------------------------------------------
mae_table = mae_part.T
mae_table.insert(0, "Ensemble", [mae_global[m] for m in mae_table.index])

mae_table = mae_table[
    ["Ensemble","1990-1999","2000-2008","2009-2019","2020-end"]
]

mae_table = mae_table.round(3)

mae_table

C:\Users\Mita\AppData\Local\Temp\ipykernel_1948\1236162249.py:54: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("partition")[["AE_LR","AE_RIDGE","AE_LGBM"]]


partition,Ensemble,1990-1999,2000-2008,2009-2019,2020-end
LR,0.680,0.421,0.378,0.597,1.777
RIDGE,0.693,0.425,0.401,0.590,1.831
LGBM,0.649,0.389,0.357,0.563,1.736


In [17]:
import pandas as pd

# -----------------------------
# 1) Accès direct au meta LGBM
# -----------------------------
meta_lgbm = meta_models["metas"]["LGBM"]

print("LGBM meta keys:", list(meta_lgbm.keys()))
print("tunable:", meta_lgbm.get("tunable"))
print("search:", meta_lgbm.get("search"), "| n_iter:", meta_lgbm.get("n_iter"))
print("tune_every_months:", meta_lgbm.get("tune_every_months"), "| tune_cv_windows:", meta_lgbm.get("tune_cv_windows"))
print("pi_windows:", meta_lgbm.get("pi_windows"), "| partitions:", meta_lgbm.get("partitions"))

# -----------------------------
# 2) Historique hyperparamètres (par bloc)
# -----------------------------
df_lgbm_params = pd.DataFrame(meta_lgbm.get("params_history", []))
if len(df_lgbm_params):
    df_lgbm_params = df_lgbm_params.sort_values("block").reset_index(drop=True)

    # garder colonnes utiles (selon ta grille)
    cols = [
        "block", "cutoff_start", "n_windows",
        "n_estimators", "max_depth", "num_leaves",
        "subsample", "colsample_bytree",
        "reg_alpha", "reg_lambda",
        "min_child_samples", "min_split_gain",
    ]
    df_lgbm_params = df_lgbm_params[[c for c in cols if c in df_lgbm_params.columns]]

print("\n=== LGBM params_history (head) ===")
print(df_lgbm_params.head(12))

# -----------------------------
# 3) Historique MAE de tuning (par bloc)
# -----------------------------
df_lgbm_tune = pd.DataFrame(meta_lgbm.get("tune_history", []))
if len(df_lgbm_tune):
    df_lgbm_tune = df_lgbm_tune.sort_values("block").reset_index(drop=True)

print("\n=== LGBM tune_history (head) ===")
print(df_lgbm_tune.head(12))

# -----------------------------
# 4) Join params + tune_mae (table finale)
# -----------------------------
if len(df_lgbm_params) and len(df_lgbm_tune):
    df_lgbm_info = df_lgbm_params.merge(
        df_lgbm_tune[["block", "tune_mae"]],
        on="block",
        how="left"
    )
else:
    df_lgbm_info = df_lgbm_params.copy()

print("\n=== LGBM info (params + tune_mae) ===")
print(df_lgbm_info.head(12))

LGBM meta keys: ['model', 'h', 'step_size', 'exp_start', 'exp_end', 'cutoff_start', 'cutoff_end', 'partitions', 'pi_windows', 'tune_every_months', 'tune_cv_windows', 'tunable', 'search', 'n_iter', 'use_conformal_in_tune', 'param_space', 'params_history', 'tune_history']
tunable: True
search: random | n_iter: 12
tune_every_months: 36 | tune_cv_windows: 6
pi_windows: 3 | partitions: 428

=== LGBM params_history (head) ===
    block cutoff_start  n_windows  n_estimators  max_depth  num_leaves
0       1   1989-01-01         36           100          5           7
1       2   1992-01-01         36            20          3          31
2       3   1995-01-01         36           100          3          15
3       4   1998-01-01         36            20          2           7
4       5   2001-01-01         36            20          2           7
5       6   2004-01-01         36            50          5           7
6       7   2007-01-01         36            50          2          15
7       

In [ ]:
import pandas as pd

ridge_params = meta_lr_ridge["metas"]["RIDGE"]["params_history"]

df_ridge_params = pd.DataFrame(ridge_params)

# garder colonnes utiles
df_ridge_params = df_ridge_params[[
    "block",
    "cutoff_start",
    "alpha"
]].sort_values("block").reset_index(drop=True)

df_ridge_params

,block,cutoff_start,alpha
0,1,1989-01-01,1.00
1,2,1992-01-01,10.00
2,3,1995-01-01,1.00
3,4,1998-01-01,0.10
4,5,2001-01-01,10.00
5,6,2004-01-01,100.00
6,7,2007-01-01,10.00
7,8,2010-01-01,0.01
8,9,2013-01-01,1.00
9,10,2016-01-01,1.00


# Tranformer Backteting

In [ ]:
bkt_score = bkt_lr_ridge_final.copy()
bkt_score["ds"] = pd.to_datetime(bkt_score["ds"], errors="coerce")

# enlever timezone si jamais
if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):
    bkt_score["ds"] = bkt_score["ds"].dt.tz_convert(None)

bkt_score = bkt_score.dropna(subset=["ds"])
bkt_score = bkt_score[(bkt_score["ds"] >= EXP_START) & (bkt_score["ds"] <= EXP_END)].reset_index(drop=True)

bins = pd.to_datetime(["1990-01-01","2000-01-01","2009-01-01","2020-01-01","2025-09-01"])
labels = ["1990-1999", "2000-2008", "2009-2019", "2020-end"]

bkt_score["partition"] = pd.cut(bkt_score["ds"], bins=bins, labels=labels, right=False, include_lowest=True)
bkt_score = bkt_score.dropna(subset=["partition"]).reset_index(drop=True)

print(bkt_score[["ds","cutoff","partition"]].head(10))
print(bkt_score["partition"].value_counts().sort_index())

          ds     cutoff  partition
0 1990-01-01 1989-12-01  1990-1999
1 1990-02-01 1990-01-01  1990-1999
2 1990-03-01 1990-02-01  1990-1999
3 1990-04-01 1990-03-01  1990-1999
4 1990-05-01 1990-04-01  1990-1999
5 1990-06-01 1990-05-01  1990-1999
6 1990-07-01 1990-06-01  1990-1999
7 1990-08-01 1990-07-01  1990-1999
8 1990-09-01 1990-08-01  1990-1999
9 1990-10-01 1990-09-01  1990-1999
partition
1990-1999    120
2000-2008    108
2009-2019    132
2020-end      68
Name: count, dtype: int64


C:\Users\Mita\AppData\Local\Temp\ipykernel_16156\2163380052.py:5: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):


# Building Leaderbord

In [ ]:
tmp = bkt_score.copy()

models = ["LR", "RIDGE", "LGBM"]

# 1) S'assurer que lower <= upper (sécurité)
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"
    tmp[[lo, hi]] = np.sort(tmp[[lo, hi]].to_numpy(), axis=1)

# 2) Wide -> Long + features de scoring
rows = []
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"

    s = tmp[["unique_id", "ds", "cutoff", "y", "partition"]].copy()
    s["model_label"] = m
    s["model_name"]  = m

    s["forecast"] = tmp[m]
    s["lower"]    = tmp[lo]
    s["upper"]    = tmp[hi]

    s["abs_err"]   = (s["y"] - s["forecast"]).abs()
    s["covered"]   = ((s["y"] >= s["lower"]) & (s["y"] <= s["upper"])).astype(int)
    s["int_width"] = (s["upper"] - s["lower"]).abs()

    rows.append(s)

long_sc = pd.concat(rows, ignore_index=True)

# 3) FIX pandas: partition en string + groupby reset_index
long_sc["partition"] = long_sc["partition"].astype(str)
long_sc = long_sc.loc[:, ~long_sc.columns.duplicated()]

score_df = (
    long_sc
    .groupby(["unique_id", "model_label", "model_name", "partition"], observed=True)
    .agg(
        mae=("abs_err", "mean"),
        coverage=("covered", "mean"),
        width=("int_width", "mean"),
        n=("y", "size"),
    )
    .reset_index()
)

# 4) Top 3 par partition
leaderboard = (
    score_df.sort_values(
        by=["partition", "mae", "coverage", "width"],
        ascending=[True, True, False, True],
    )
    .groupby("partition", as_index=False)
    .head(3)
)

print(score_df.sort_values(["partition", "mae"]).head(20))
print("\nTop 3 par partition:")
print(leaderboard)

KeyError: "None of [Index(['LGBM-lo-95', 'LGBM-hi-95'], dtype='object')] are in the [columns]"

# MLFLOW

In [ ]:
import mlflow

# =====================================================
# 0) (Optionnel mais recommandé) Tracking URI
# =====================================================
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# =====================================================
# 1) Set experiment
# =====================================================
experiment_name = "Multivariate_experiment_design"
mlflow.set_experiment(experiment_name)

# =====================================================
# 2) DataFrame à logger
# =====================================================
df_log = score_df.copy()  # ou leaderboard_global

# =====================================================
# 3) Logging loop
# =====================================================
for idx, row in df_log.iterrows():
    
    partition = row["partition"] if "partition" in df_log.columns else "global"
    run_name = f"{row['model_label']}_partition_{partition}"
    
    with mlflow.start_run(run_name=run_name):
        
        # ======================
        # PARAMETERS
        # ======================
        mlflow.log_param("model_label", row["model_label"])
        mlflow.log_param("model_name", row["model_name"])
        mlflow.log_param("partition", partition)
        mlflow.log_param("horizon", 12)
        mlflow.log_param("lags", list(range(1, 25)))
        mlflow.log_param("date_features", ["year", "month", "quarter"])
        mlflow.log_param("conformal_method", "conformal_distribution")
        
        # ======================
        # METRICS
        # ======================
        mlflow.log_metric("mae", float(row["mae"]))
        mlflow.log_metric("coverage", float(row["coverage"]))
        mlflow.log_metric("width", float(row["width"]))
        
        # ======================
        # TAGS (optionnel mais pro)
        # ======================
        mlflow.set_tag("project", "UNRATE_forecasting")
        mlflow.set_tag("framework", "MLForecast")
        mlflow.set_tag("model_family", row["model_name"])

print("Logging terminé.")

# Vérif

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

# =========================================================
# MAE pour LR (global + par partition) à partir de bkt_lr_final
# Colonnes attendues: unique_id, ds, y, LR
# =========================================================

EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")

df = bkt_lr_final.copy()

# ds propre
df["ds"] = pd.to_datetime(df["ds"], errors="coerce")
if pd.api.types.is_datetime64tz_dtype(df["ds"]):
    df["ds"] = df["ds"].dt.tz_convert(None)

# garder lignes valides
df = df.dropna(subset=["ds", "y", "LR"]).copy()
df = df[(df["ds"] >= EXP_START) & (df["ds"] <= EXP_END)].reset_index(drop=True)

# -----------------------------
# MAE global
# -----------------------------
mae_all = mean_absolute_error(df["y"], df["LR"])
print("✅ MAE moyen (LR) — GLOBAL :", float(mae_all))

# -----------------------------
# MAE par partition macro (based on ds)
# -----------------------------
bins = pd.to_datetime(["1990-01-01","2000-01-01","2009-01-01","2020-01-01","2025-09-01"])
labels = ["1990-1999", "2000-2008", "2009-2019", "2020-fin"]

df["partition"] = pd.cut(df["ds"], bins=bins, labels=labels, right=False, include_lowest=True)
df = df.dropna(subset=["partition"]).copy()

mae_by_partition = (
    df.assign(abs_err=np.abs(df["y"] - df["LR"]))
      .groupby("partition")["abs_err"]
      .mean()
      .sort_index()
)

n_by_partition = df.groupby("partition").size().sort_index()

print("\n✅ MAE moyen (LR) — PAR PARTITION :")
print(mae_by_partition)

print("\n✅ N obs — PAR PARTITION :")
print(n_by_partition)

# (optionnel) MAE pondéré (≈ MAE global)
mae_weighted = float((mae_by_partition * n_by_partition).sum() / n_by_partition.sum())
print("\n✅ MAE moyen (LR) — pondéré partitions :", mae_weighted)

✅ MAE moyen (LR) — GLOBAL : 0.6799697179008874

✅ MAE moyen (LR) — PAR PARTITION :
partition
1990-1999    0.421220
2000-2008    0.377789
2009-2019    0.597483
2020-fin     1.776643
Name: abs_err, dtype: float64

✅ N obs — PAR PARTITION :
partition
1990-1999    120
2000-2008    108
2009-2019    132
2020-fin      68
dtype: int64

✅ MAE moyen (LR) — pondéré partitions : 0.6799697179008874


C:\Users\Mita\AppData\Local\Temp\ipykernel_16388\3500170289.py:17: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df["ds"]):
C:\Users\Mita\AppData\Local\Temp\ipykernel_16388\3500170289.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("partition")["abs_err"]
C:\Users\Mita\AppData\Local\Temp\ipykernel_16388\3500170289.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  n_by_partition = df.groupby("partition").size().sort_index()
